# Fake Review Detection - Model Training

In [2]:
# Import all required packages

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import os


In [ ]:
# Load train and test datasets from the Data folder

data_dir = 'Data'

df_train = pd.read_csv(os.path.join(data_dir, 'train.csv'), sep=';')
df_test = pd.read_csv(os.path.join(data_dir, 'test.csv'), sep=';')

print(f'Train: {len(df_train)}, Test: {len(df_test)}')

## Prepare Features

In [ ]:
# Separate raw text (features) from labels

X_train = df_train['text']
y_train = df_train['ai_generated']

X_test = df_test['text']
y_test = df_test['ai_generated']

## TF-IDF Vectorization

In [ ]:
# Convert raw review text into TF-IDF feature vectors (fit on train only)

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## Hyperparameter Tuning + Logistic Regression

In [ ]:
# Tune Logistic Regression hyperparameters using 5-fold cross-validation

from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': ['balanced']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_tfidf, y_train)
model = grid.best_estimator_
print(f'Best params: {grid.best_params_}')
print(f'Best CV F1:  {grid.best_score_:.4f}')
print(f'Vocabulary size: {len(tfidf.get_feature_names_out())}')

## Evaluation on Train Set

In [ ]:
# Evaluate the tuned model on the training set

y_pred_train = model.predict(X_train_tfidf)

print('=== Train Set ===')
print(f'Accuracy:  {accuracy_score(y_train, y_pred_train):.4f}')
print(f'F1-Score:  {f1_score(y_train, y_pred_train):.4f}')
print(f'Precision: {precision_score(y_train, y_pred_train):.4f}')
print(f'Recall:    {recall_score(y_train, y_pred_train):.4f}')

## Evaluation on Test Set

In [ ]:
# Evaluate the tuned model on the held-out test set

y_pred = model.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)

print('=== Test Set ===')
print(f'Accuracy:  {acc:.4f}')
print(f'F1-Score:  {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred))

from sklearn.metrics import roc_auc_score
y_prob = model.predict_proba(X_test_tfidf)[:, 1]
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}')

## Top 10 Markers for AI (Fake) and Human (Real)

In [ ]:
# Show the top 10 most predictive words/n-grams for each class

feature_names = np.array(tfidf.get_feature_names_out())
coefs = model.coef_[0]
top_ai_idx = np.argsort(coefs)[-10:][::-1]
top_human_idx = np.argsort(coefs)[:10]

print("=== Top 10 Markers for AI-Generated (Fake) ===")
for i in top_ai_idx:
    print(f"  {feature_names[i]:25s} weight: {coefs[i]:.4f}")

print("\n=== Top 10 Markers for Human-Written (Real) ===")
for i in top_human_idx:
    print(f"  {feature_names[i]:25s} weight: {coefs[i]:.4f}")

## Leave-One-Category-Out Evaluation

In [ ]:
# Leave-One-Category-Out (LOCO) cross-domain evaluation
# For each product category: train on all other categories, test on the held-out one.
# This tests whether the model generalizes to product types it has never seen.

from sklearn.metrics import precision_recall_fscore_support

results_loc = []
categories = sorted(df_train['category'].unique())

for held_out in categories:
    # Split: train on everything except the held-out category
    train_sub = df_train[df_train['category'] != held_out]
    test_sub = df_test[df_test['category'] == held_out]

    # Vectorize text with a fresh TF-IDF (fit only on this training subset)
    tfidf_loc = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
    X_tr = tfidf_loc.fit_transform(train_sub['text'])
    X_te = tfidf_loc.transform(test_sub['text'])

    # Train Logistic Regression with the best hyperparameters from tuning
    m = LogisticRegression(C=10, class_weight='balanced', solver='liblinear', max_iter=1000, random_state=42)
    m.fit(X_tr, train_sub['ai_generated'])
    pred = m.predict(X_te)

    # Store metrics for this category
    results_loc.append({
        'category': held_out,
        'accuracy': accuracy_score(test_sub['ai_generated'], pred),
        'precision': precision_score(test_sub['ai_generated'], pred, zero_division=0),
        'recall': recall_score(test_sub['ai_generated'], pred, zero_division=0),
        'f1': f1_score(test_sub['ai_generated'], pred, zero_division=0),
        'samples': len(test_sub)
    })

# Build results table and add an AVERAGE row
loc_df = pd.DataFrame(results_loc)
avg_row = pd.DataFrame([{
    'category': 'AVERAGE',
    'accuracy': loc_df['accuracy'].mean(),
    'precision': loc_df['precision'].mean(),
    'recall': loc_df['recall'].mean(),
    'f1': loc_df['f1'].mean(),
    'samples': loc_df['samples'].sum()
}])
pd.concat([loc_df, avg_row], ignore_index=True)